# Mammography Exercise

**Notebook 6 of 6, and optional.** Everything from notebooks 01 to 05, on a new problem:
**screening mammograms.** Each row is a small region of a mammogram, an X-ray image of a
breast, described by six numbers computed from the image. The target `calcification` is
1 where the region shows a microcalcification, a tiny calcium deposit that can be an early
sign of breast cancer and is followed up by a doctor.

About 2 regions in 100 are positive. This time a missed case is clearly the worse mistake,
and the problem turns out to be harder than the fraud data.

Each step has a cell marked `# your code`. Work top to bottom.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, fbeta_score,
    confusion_matrix, precision_recall_curve, roc_auc_score, average_precision_score,
)

In [ ]:
df = pd.read_csv("../data/mammography.csv")
df.head()

The six features come without names, only as image measurements, and they are already
standardized to mean 0 and standard deviation 1.

## Step 1: The imbalance and the split

How many regions are positive, as a count and as a share? Then separate features and
target, and split with `test_size=0.25`, `random_state=42` and stratification.

In [ ]:
print(df["calcification"].value_counts())
print(f"share positive: {df['calcification'].mean():.3f}")

X = df.drop(columns="calcification")
y = df["calcification"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("positives in the test set:", y_test.sum(), "of", len(y_test))

## Step 2: The do-nothing baseline

What test accuracy does "never a calcification" reach, and what is its recall?

In [ ]:
y_pred_baseline = np.zeros(len(y_test), dtype=int)

print(f"accuracy: {accuracy_score(y_test, y_pred_baseline):.3f}")
print(f"recall:   {recall_score(y_test, y_pred_baseline):.3f}")

## Step 3: A plain logistic regression

Scale the features, train `LogisticRegression(max_iter=1000)`, and report its confusion
matrix, accuracy, precision, recall and F2 on the test set.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred))
print(f"accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"precision: {precision_score(y_test, y_pred):.3f}")
print(f"recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F2:        {fbeta_score(y_test, y_pred, beta=2):.3f}")

> **Exercise:**
>
> No code needed. The model is 98 percent accurate. Would you let it screen mammograms?
> Which of the numbers above decides, and why?

Your answer here:

<details><summary>
Click here to compare your answer...
</summary>

No. **Recall is about 0.37**: the model misses almost two calcifications in three, and its
accuracy is barely above the 97.7 percent of doing nothing. For screening, recall is the
number that decides, with F2 as the single score, since a missed case is far worse than a
region a doctor looks at and dismisses.
</details>

## Step 4: Class weights

Train the same model with `class_weight="balanced"` and report precision, recall and F2.

In [ ]:
model_weighted = LogisticRegression(max_iter=1000, class_weight="balanced")
model_weighted.fit(X_train_scaled, y_train)
pred_weighted = model_weighted.predict(X_test_scaled)

print(f"precision: {precision_score(y_test, pred_weighted):.3f}")
print(f"recall:    {recall_score(y_test, pred_weighted):.3f}")
print(f"F2:        {fbeta_score(y_test, pred_weighted, beta=2):.3f}")

## Step 5: A threshold for a target recall

The screening team asks for a recall of **at least 0.8**. Choose the highest threshold
that reaches it **on the training set**, using the plain model's probabilities and
`precision_recall_curve` as in notebook 04. Then apply it to the test set once.

In [ ]:
proba_train = model.predict_proba(X_train_scaled)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_train, proba_train)
candidates = pd.DataFrame(
    {"threshold": thresholds, "precision": precision[:-1], "recall": recall[:-1]}
)
chosen = candidates[candidates["recall"] >= 0.8]["threshold"].max()

proba_test = model.predict_proba(X_test_scaled)[:, 1]
pred_chosen = (proba_test >= chosen).astype(int)

print(f"chosen threshold: {chosen:.3f}")
print(f"precision: {precision_score(y_test, pred_chosen):.3f}")
print(f"recall:    {recall_score(y_test, pred_chosen):.3f}")
print(f"F2:        {fbeta_score(y_test, pred_chosen, beta=2):.3f}")

<details><summary>
Click here to compare your answer...
</summary>

The threshold is very low, about **0.04**, and on the test set recall is about **0.88**,
above the target this time, with precision about 0.28. Class weights reached a similar
recall, 0.86, but at a precision of 0.18. The chosen threshold gives the better trade and
the higher F2.

A precision of 0.28 means nearly four regions sent to a doctor for every real
calcification. For screening that is usually acceptable; a missed one is not.
</details>

## Step 6: The two summaries

Compute the plain model's ROC AUC and average precision on the test set, and compare each
with what random guessing would score.

In [ ]:
print(f"ROC AUC:           {roc_auc_score(y_test, proba_test):.3f}   random guessing: 0.5")
print(f"average precision: {average_precision_score(y_test, proba_test):.3f}   random guessing: {y_test.mean():.3f}")

<details><summary>
Click here to compare your answer...
</summary>

A ROC AUC of about **0.96** suggests an excellent model. An average precision of about
**0.62** says what steps 3 to 5 showed the hard way: finding the rare positives without
many false alarms is difficult here. The gap between the two is wider than on the fraud
data, because this problem is harder and the imbalance stronger.
</details>

## Summary

| Recipe | Precision | Recall | F2 |
| :-- | --: | --: | --: |
| Never a calcification | 0 | 0 | 0 |
| Plain model, threshold 0.5 | 0.75 | 0.37 | 0.41 |
| Class weights | 0.18 | 0.86 | 0.49 |
| Threshold chosen for recall 0.8 | 0.28 | 0.88 | 0.61 |

The same steps as on the fraud data, and a different problem. Accuracy said 98 percent;
recall said the model missed most cases; a metric chosen from the cost of each mistake,
and a threshold chosen on the training set, turned it into something a screening team
could use.